## 1. Import required Libraries

In [25]:
import os
import numpy as np 
import tensorflow as tf 
from tensorflow.keras import layers, models 
from tensorflow.keras.datasets import cifar10 
from tensorflow.keras.utils import to_categorical 
import matplotlib.pyplot as plt 
from xgboost import XGBClassifier 
from sklearn import preprocessing 
from sklearn.model_selection import cross_val_predict 
from sklearn. model_selection import train_test_split 
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder

## 2. Loading the Data

In [26]:

# Directory containing image subfolders
image_dir = './CHS2406_Coursework2_Data_Repository'
labels = []
data = []

# Allowed image extensions (make sure to use correct ones)
allowed_extensions = {'.png', '.jpg', '.jpeg', '.JPG'}

# Target size for resizing images (modify based on your needs)
img_size = (150, 150,3)  # Image size (height, width)

for label in os.listdir(image_dir):
    subfolder_path = os.path.join(image_dir, label)
    
    if os.path.isdir(subfolder_path):
        # Assign label based on folder name
        for image_file in os.listdir(subfolder_path):
            image_path = os.path.join(subfolder_path, image_file)
            
            # Check if the file has an allowed extension
            if any(image_file.lower().endswith(ext) for ext in allowed_extensions):
                try:
                    # Load the image with the specified target size and RGB mode
                    image = tf.keras.preprocessing.image.load_img(image_path, color_mode='rgb', target_size=img_size)
                    
                    # Convert the image to a NumPy array
                    image = np.array(image)
                    
                    # Append the label and image to the respective lists
                    labels.append(int(label[-1])) #From the string 'stage1' it takes the last digit and converts it to int
                    data.append(image)  # Append the image itself
                
                except Exception as e:
                    e


your the best ./CHS2406_Coursework2_Data_Repository/Stage8/Stage_8_U2265620(4).jpeg: cannot identify image file <_io.BytesIO object at 0x1095556c0>
your the best ./CHS2406_Coursework2_Data_Repository/Stage8/Stage_8_U2265620(6).jpeg: cannot identify image file <_io.BytesIO object at 0x307a24810>
your the best ./CHS2406_Coursework2_Data_Repository/Stage8/Stage_8_U2265620(2).jpeg: cannot identify image file <_io.BytesIO object at 0x306556c50>
your the best ./CHS2406_Coursework2_Data_Repository/Stage8/Stage_8_U2265620(7).jpeg: cannot identify image file <_io.BytesIO object at 0x307a4c0e0>
your the best ./CHS2406_Coursework2_Data_Repository/Stage8/Stage_8_U2265620(5).jpeg: cannot identify image file <_io.BytesIO object at 0x1095556c0>
your the best ./CHS2406_Coursework2_Data_Repository/Stage8/Stage_8_U2265620(1).jpeg: cannot identify image file <_io.BytesIO object at 0x1095556c0>
your the best ./CHS2406_Coursework2_Data_Repository/Stage8/Stage_8_U2265620(3).jpeg: cannot identify image file 

## 3. Data Labelling Errors

In [27]:
data = np.array(data)
labels = np.array(labels)

In [28]:
encoder = OneHotEncoder(sparse_output=False)
ohe_labels = labels.reshape(-1,1)
ohe_labels = encoder.fit_transform(ohe_labels)
ohe_labels

array([[0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]])

In [29]:
normalized_data = data / 255.0

In [30]:
train_data, test_data, train_labels, test_labels = train_test_split(normalized_data,ohe_labels,random_state=42)

In [34]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150,150,3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(8, activation='softmax')

])

In [35]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [36]:
model.fit(train_data, train_labels, epochs= 10, batch_size= 64, validation_data=(test_data, test_labels))

Epoch 1/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 71s 474ms/step - accuracy: 0.1307 - loss: 2.1691 - val_accuracy: 0.1456 - val_loss: 2.0722
Epoch 2/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 65s 440ms/step - accuracy: 0.2013 - loss: 2.0247 - val_accuracy: 0.2381 - val_loss: 2.0025
Epoch 3/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 65s 437ms/step - accuracy: 0.3987 - loss: 1.6746 - val_accuracy: 0.2845 - val_loss: 2.0410
Epoch 4/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 64s 434ms/step - accuracy: 0.6549 - loss: 0.9963 - val_accuracy: 0.3004 - val_loss: 2.5346
Epoch 5/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 67s 451ms/step - accuracy: 0.8667 - loss: 0.4360 - val_accuracy: 0.3048 - val_loss: 4.1047
Epoch 6/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 64s 434ms/step - accuracy: 0.9547 - loss: 0.1770 - val_accuracy: 0.3071 - val_loss: 4.9602
Epoch 7/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 64s 435ms/step - accuracy: 0.9808 - loss: 0.0943 - val_accuracy: 0.3026 - val_loss: 5.3537
Epoch 8/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 68s 457ms/step - accuracy: 0.9891 - loss: 0

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data,test_labels,verbose=2)
print(f"Accuracy: {test_accuracy:.2f}")

99/99 - 5s - 54ms/step - accuracy: 0.1179 - loss: 2.1403
Accuracy: 0.12


## Label errors:

<ol>
  <li>Explain what kind of errors you found in the dataset.</li>
  <li>List the total number of images left in each class/stage after the label error handling</li>
</ol>

<br>

<ol>
  <li>Stage 1: <<Number of images>></li>
  <li>Stage 2: <<Number of images>></li>
  <li>Stage 3: <<Number of images>></li>
  <li>Stage 4: <<Number of images>></li>
  <li>Stage 5: <<Number of images>></li>
  <li>Stage 6: <<Number of images>></li>
  <li>Stage 7: <<Number of images>></li>
  <li>Stage 8: <<Number of images>></li>
</ol>

## 4. Pre-process the Dataset

In [12]:
# <<insert yout code here>>

## 5. Split the data
<br>

Split the data into training, validation and testing dataset using Startification, ensuring equal class distribution.

Choose appropriate values of training, validation and testing datasets.

Display total number of images in each dataset split.

In [13]:
# <<insert yout code here>>

## 6. Model Implementation

In [14]:
# <<insert yout code here>>

## 7. Evaluate the Model

In [15]:
# <<insert yout code here>>

### Training Curves

In [16]:
# <<insert yout code here>>

### Make Inference
For some unseen data, make predictions using the trained model.

In [17]:
# <<insert yout code here>>